# 06-P3. paper-aligned event evaluation

이 노트북은 point anomaly를 criticality counter로 누적해 event-wise detection rate, false alarm rate, lead time을 계산한다.

현재 구현 결정:
- point anomaly 기준은 06-P2의 `train_rmse_p099`
- validation event 기준으로 `severity-weighted criticality_threshold` sweep
- 선택 지표는 precision 우선의 `F0.5`


In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 240)


def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for path in [current, *current.parents]:
        if (path / 'pyproject.toml').exists():
            return path
    raise FileNotFoundError('pyproject.toml을 찾을 수 없습니다.')


def fbeta_score(tp: int, fp: int, fn: int, beta: float = 0.5) -> float:
    if tp == 0:
        return 0.0
    beta2 = beta * beta
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    denom = beta2 * precision + recall
    return (1 + beta2) * precision * recall / denom if denom else 0.0


def bool_value(value) -> bool:
    if pd.isna(value):
        return False
    return bool(value)


def apply_criticality_counter(event_df: pd.DataFrame, *, criticality_threshold: float) -> tuple[pd.DataFrame, dict]:
    rows = []
    counter = 0.0
    detected = False
    detection_time = pd.NaT
    max_counter = 0.0

    ordered = event_df.sort_values('window_end').copy()
    for row in ordered.itertuples(index=False):
        point_anomaly = bool(row.point_anomaly)
        maintenance_related = bool_value(getattr(row, 'maintenance_related', False))
        if point_anomaly and not maintenance_related:
            severity_increment = max(float(row.anomaly_score) - 1.0, 0.25)
            counter += severity_increment
            transition = f'increment_{severity_increment:.3f}'
        elif point_anomaly and maintenance_related:
            counter = counter
            transition = 'hold_maintenance'
        else:
            counter = max(counter - 1.0, 0.0)
            transition = 'decrement_or_floor'

        max_counter = max(max_counter, counter)
        criticality_crossed = counter >= criticality_threshold
        if (not detected) and criticality_crossed:
            detected = True
            detection_time = row.window_end

        row_dict = row._asdict()
        row_dict['criticality_counter'] = counter
        row_dict['criticality_transition'] = transition
        row_dict['criticality_threshold'] = criticality_threshold
        row_dict['criticality_crossed'] = criticality_crossed
        row_dict['is_detected_in_event'] = detected
        rows.append(row_dict)

    summary = {
        'detected': detected,
        'detection_time': detection_time,
        'max_counter': max_counter,
    }
    return pd.DataFrame(rows), summary


PROJECT_ROOT = find_project_root(Path.cwd())
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'processed' / 'paper_aligned'
SCORES_PATH = OUTPUT_DIR / 'autoencoder_reconstruction_scores.csv'
EVENT_SELECTION_PATH = OUTPUT_DIR / 'event_evaluation_windows.csv'

SUMMARY_OUTPUT = OUTPUT_DIR / 'event_detection_summary.csv'
METRICS_OUTPUT = OUTPUT_DIR / 'event_detection_metrics.csv'
TIMELINE_OUTPUT = OUTPUT_DIR / 'event_detection_timeline.csv'
METADATA_OUTPUT = OUTPUT_DIR / 'event_detection_metadata.json'

scores_df = pd.read_csv(SCORES_PATH, parse_dates=['window_start', 'window_end', 'event_start', 'event_end', 'report_date'])
event_selection_df = pd.read_csv(EVENT_SELECTION_PATH, parse_dates=['window_start', 'window_end', 'event_start', 'event_end', 'report_date'])

merge_keys = ['manufacturer', 'substation_id', 'window_start', 'window_end', 'label', 'event_type', 'event_id', 'event_split']
extra_columns = ['maintenance_related', 'disturbance_count']
base_df = scores_df.merge(
    event_selection_df[merge_keys + extra_columns],
    on=merge_keys,
    how='left',
    validate='one_to_one',
)

base_df['maintenance_related'] = base_df['maintenance_related'].fillna(False).astype(bool)
base_df['point_anomaly'] = base_df['is_above_train_p099'].astype(bool)
evaluation_df = base_df[base_df['selected_for_event_eval'] == True].copy()

CRITICALITY_THRESHOLD_CANDIDATES = [0.75, 1.0, 1.25, 1.5, 2.0]
POINT_THRESHOLD_NAME = 'train_rmse_p099'
FBETA_BETA = 0.5

metric_rows = []
for criticality_threshold in CRITICALITY_THRESHOLD_CANDIDATES:
    event_rows = []
    for event_key, event_group in evaluation_df.groupby(['manufacturer', 'event_type', 'event_split', 'event_id'], sort=False):
        manufacturer, event_type, event_split, event_id = event_key
        timeline_df, event_summary = apply_criticality_counter(event_group, criticality_threshold=criticality_threshold)
        report_date = event_group['report_date'].dropna().iloc[0] if event_group['report_date'].notna().any() else pd.NaT
        lead_time_hours = np.nan
        if event_type == 'fault_event' and event_summary['detected'] and pd.notna(report_date):
            lead_time_hours = (report_date - event_summary['detection_time']).total_seconds() / 3600.0
        event_rows.append(
            {
                'manufacturer': manufacturer,
                'event_type': event_type,
                'event_split': event_split,
                'event_id': int(event_id),
                'substation_id': int(event_group['substation_id'].iloc[0]),
                'configuration_type': event_group['configuration_type'].iloc[0],
                'fault_label': event_group['fault_label'].iloc[0],
                'report_date': report_date,
                'event_start': event_group['event_start'].iloc[0],
                'event_end': event_group['event_end'].iloc[0],
                'window_count': int(len(event_group)),
                'point_anomaly_count': int(event_group['point_anomaly'].sum()),
                'max_counter': float(event_summary['max_counter']),
                'detected': bool(event_summary['detected']),
                'detection_time': event_summary['detection_time'],
                'lead_time_hours': lead_time_hours,
                'criticality_threshold': criticality_threshold,
            }
        )

    event_metrics_df = pd.DataFrame(event_rows)
    for split in ['validation', 'holdout']:
        split_df = event_metrics_df[event_metrics_df['event_split'] == split].copy()
        if split_df.empty:
            continue
        fault_events = split_df[split_df['event_type'] == 'fault_event'].copy()
        normal_events = split_df[split_df['event_type'] == 'normal_event'].copy()
        tp = int(fault_events['detected'].sum())
        fn = int((~fault_events['detected']).sum())
        fp = int(normal_events['detected'].sum())
        tn = int((~normal_events['detected']).sum())
        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        false_alarm_rate = fp / (fp + tn) if (fp + tn) else 0.0
        event_recognition_rate = tn / (fp + tn) if (fp + tn) else 0.0
        pointwise_normal_accuracy = float(
            evaluation_df[
                evaluation_df['event_split'].eq(split)
                & evaluation_df['event_type'].eq('normal_event')
            ]['point_anomaly'].eq(False).mean()
        )
        detected_faults = fault_events[fault_events['detected']].copy()
        metric_rows.append(
            {
                'criticality_threshold': criticality_threshold,
                'split': split,
                'point_threshold_name': POINT_THRESHOLD_NAME,
                'event_fault_count': int(len(fault_events)),
                'event_normal_count': int(len(normal_events)),
                'tp': tp,
                'fn': fn,
                'fp': fp,
                'tn': tn,
                'precision': precision,
                'recall': recall,
                'fbeta_05': fbeta_score(tp, fp, fn, beta=FBETA_BETA),
                'false_alarm_rate': false_alarm_rate,
                'normal_event_recognition_rate': event_recognition_rate,
                'normal_pointwise_accuracy': pointwise_normal_accuracy,
                'avg_lead_time_hours_detected_faults': float(detected_faults['lead_time_hours'].mean()) if not detected_faults.empty else np.nan,
            }
        )

metrics_df = pd.DataFrame(metric_rows)
validation_metrics = metrics_df[metrics_df['split'] == 'validation'].copy()
validation_metrics = validation_metrics.sort_values(
    ['fbeta_05', 'precision', 'recall', 'false_alarm_rate'],
    ascending=[False, False, False, True],
)
SELECTED_CRITICALITY_THRESHOLD = float(validation_metrics.iloc[0]['criticality_threshold'])

timeline_frames = []
summary_rows = []
for event_key, event_group in evaluation_df.groupby(['manufacturer', 'event_type', 'event_split', 'event_id'], sort=False):
    manufacturer, event_type, event_split, event_id = event_key
    timeline_df, event_summary = apply_criticality_counter(event_group, criticality_threshold=SELECTED_CRITICALITY_THRESHOLD)
    timeline_df['selected_criticality_threshold'] = SELECTED_CRITICALITY_THRESHOLD
    timeline_frames.append(timeline_df)
    report_date = event_group['report_date'].dropna().iloc[0] if event_group['report_date'].notna().any() else pd.NaT
    lead_time_hours = np.nan
    if event_type == 'fault_event' and event_summary['detected'] and pd.notna(report_date):
        lead_time_hours = (report_date - event_summary['detection_time']).total_seconds() / 3600.0
    summary_rows.append(
        {
            'manufacturer': manufacturer,
            'event_type': event_type,
            'event_split': event_split,
            'event_id': int(event_id),
            'substation_id': int(event_group['substation_id'].iloc[0]),
            'configuration_type': event_group['configuration_type'].iloc[0],
            'fault_label': event_group['fault_label'].iloc[0],
            'report_date': report_date,
            'event_start': event_group['event_start'].iloc[0],
            'event_end': event_group['event_end'].iloc[0],
            'window_count': int(len(event_group)),
            'point_anomaly_count': int(event_group['point_anomaly'].sum()),
            'max_counter': float(event_summary['max_counter']),
            'detected': bool(event_summary['detected']),
            'detection_time': event_summary['detection_time'],
            'lead_time_hours': lead_time_hours,
            'selected_criticality_threshold': SELECTED_CRITICALITY_THRESHOLD,
        }
    )

timeline_output = pd.concat(timeline_frames, ignore_index=True)
summary_df = pd.DataFrame(summary_rows)
metrics_df['selected_threshold'] = metrics_df['criticality_threshold'].eq(SELECTED_CRITICALITY_THRESHOLD)

timeline_output.to_csv(TIMELINE_OUTPUT, index=False, encoding='utf-8-sig')
summary_df.to_csv(SUMMARY_OUTPUT, index=False, encoding='utf-8-sig')
metrics_df.to_csv(METRICS_OUTPUT, index=False, encoding='utf-8-sig')

metadata = {
    'selected_criticality_threshold': SELECTED_CRITICALITY_THRESHOLD,
    'point_threshold_name': POINT_THRESHOLD_NAME,
    'point_threshold_source': '06-P2 train_rmse_p099',
    'criticality_counter_rule': 'increment by max(anomaly_score - 1.0, 0.25) on anomaly during expected normal operation, decrement by one on non-anomaly, hold during maintenance anomaly, floor at zero',
    'counter_mode': 'severity_weighted',
    'selection_metric': 'validation event-wise F0.5',
    'beta': FBETA_BETA,
    'limitation': 'fault and normal event evaluation windows are clipped to a fixed 7-day window before the event reference end; short source events keep only the available windows within that interval',
}
METADATA_OUTPUT.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8')

display(validation_metrics.head(10))
display(metrics_df[metrics_df['selected_threshold'] == True].sort_values('split'))
display(summary_df.groupby(['event_type', 'event_split'])['detected'].sum().reset_index())
print(f'selected criticality threshold: {SELECTED_CRITICALITY_THRESHOLD}')
print(f'saved: {TIMELINE_OUTPUT}')
print(f'saved: {SUMMARY_OUTPUT}')
print(f'saved: {METRICS_OUTPUT}')
print(f'saved: {METADATA_OUTPUT}')


,criticality_threshold,split,point_threshold_name,event_fault_count,event_normal_count,tp,fn,fp,tn,precision,recall,fbeta_05,false_alarm_rate,normal_event_recognition_rate,normal_pointwise_accuracy,avg_lead_time_hours_detected_faults
2,1.00,validation,train_rmse_p099,12,10,5,7,0,10,1.000000,0.416667,0.781250,0.0,1.0,0.989286,59.763333
4,1.25,validation,train_rmse_p099,12,10,5,7,0,10,1.000000,0.416667,0.781250,0.0,1.0,0.989286,59.763333
6,1.50,validation,train_rmse_p099,12,10,4,8,0,10,1.000000,0.333333,0.714286,0.0,1.0,0.989286,56.066667
0,0.75,validation,train_rmse_p099,12,10,5,7,1,9,0.833333,0.416667,0.694444,0.1,0.9,0.989286,60.963333
8,2.00,validation,train_rmse_p099,12,10,3,9,0,10,1.000000,0.250000,0.625000,0.0,1.0,0.989286,51.833333


,criticality_threshold,split,point_threshold_name,event_fault_count,event_normal_count,tp,fn,fp,tn,precision,recall,fbeta_05,false_alarm_rate,normal_event_recognition_rate,normal_pointwise_accuracy,avg_lead_time_hours_detected_faults,selected_threshold
3,1.0,holdout,train_rmse_p099,11,11,4,7,0,11,1.0,0.363636,0.740741,0.0,1.0,0.973684,31.762500,True
2,1.0,validation,train_rmse_p099,12,10,5,7,0,10,1.0,0.416667,0.781250,0.0,1.0,0.989286,59.763333,True


,event_type,event_split,detected
0,fault_event,holdout,4
1,fault_event,validation,5
2,normal_event,holdout,0
3,normal_event,validation,0


selected criticality threshold: 1.0
saved: C:\Project3\HeatGrid_Agent\data\processed\paper_aligned\event_detection_timeline.csv
saved: C:\Project3\HeatGrid_Agent\data\processed\paper_aligned\event_detection_summary.csv
saved: C:\Project3\HeatGrid_Agent\data\processed\paper_aligned\event_detection_metrics.csv
saved: C:\Project3\HeatGrid_Agent\data\processed\paper_aligned\event_detection_metadata.json


## 다음 단계

- `event_detection_timeline.csv`를 기반으로 06-P4에서 주요 이상 feature 설명을 생성한다.
- `event_detection_summary.csv`는 06-P5에서 `risk_score`, `risk_level`, `priority_score` 변환의 직접 입력으로 사용한다.
